In [3]:
import openai
print(openai.__version__)

2.15.0


In [5]:
import pandas as pd
import numpy as np
import requests

In [6]:
df = pd.read_csv("../data/milestone1_output_suresh.csv")
df.head(2)

,contract_id,extracted_text,apr,term_months,monthly_payment,penalty
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,none
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,early termination fee $300


In [8]:
# ================================
# Week 3 – LLM Prompt Design
# ================================

SLA_PROMPT_TEMPLATE = """
You are a legal contract analysis assistant specialized in car lease agreements.

Extract the following SLA details from the contract text.
If a value is missing, return null.
Return ONLY valid JSON. No explanation.

Fields:
- interest_rate_apr
- lease_term_months
- monthly_payment
- down_payment
- residual_value
- mileage_allowance
- overage_charge
- early_termination
- purchase_option
- maintenance_responsibility
- warranty_insurance
- penalties
- missing_or_ambiguous_clauses

Contract Text:
\"\"\"
{contract_text}
\"\"\"
"""

In [9]:
# ================================
# Week 3 – OpenAI LLM Extraction
# ================================



def extract_sla_with_gpt(contract_text):
    prompt = SLA_PROMPT_TEMPLATE.format(contract_text=contract_text)

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    raw_output = response.output_text.strip()

    match = re.search(r"\{[\s\S]*\}", raw_output)
    if not match:
        return {"error": "No JSON found", "raw_output": raw_output}

    try:
        return json.loads(match.group(0))
    except Exception as e:
        return {"error": str(e), "json_text": match.group(0)}

In [10]:
test_output = extract_sla_with_gpt(df.loc[0,
"extracted_text"])
test_output

{'interest_rate_apr': 10.49,
 'lease_term_months': 24,
 'monthly_payment': 959,
 'down_payment': None,
 'residual_value': None,
 'mileage_allowance': None,
 'overage_charge': None,
 'early_termination': None,
 'purchase_option': None,
 'maintenance_responsibility': None,
 'warranty_insurance': None,
 'penalties': None,
 'missing_or_ambiguous_clauses': ['down_payment',
  'residual_value',
  'mileage_allowance',
  'overage_charge',
  'early_termination',
  'purchase_option',
  'maintenance_responsibility',
  'warranty_insurance',
  'penalties']}

In [13]:
# ================================
# Week 3 – Apply LLM & Store SLA
# ================================

sample_df = df.head(1)

sla_json_records = sample_df.apply(
    lambda row: {
        "contract_id": int(row.name),
        "sla": extract_sla_with_gpt(row["extracted_text"])
    },
    axis=1
).tolist()

sla_json_records[0]

# Store SLA JSON
with open("../data/milestone2_sla_output.json", "w") as f:
    json.dump(sla_json_records, f, indent=2)

print("SLA JSON stored successfully using OpenAI GPT")

SLA JSON stored successfully using OpenAI GPT


In [14]:
# ===============================
# Week 3 – Accuracy Evaluation (Sample Only)
# ===============================

# Use only 1 sample contract to avoid rate limits
eval_df = df.head(1).copy()

# Apply LLM extraction (SAFE – only 1 call)
eval_df["llm_sla"] = eval_df["extracted_text"].apply(extract_sla_with_gpt)

eval_df[["llm_sla"]] 

,llm_sla
0,"{'interest_rate_apr': '10.49%', 'lease_term_mo..."


In [15]:
# Extract individual SLA fields from LLM output
eval_df["llm_apr"] = eval_df["llm_sla"].apply(lambda x: x.get("interest_rate_apr"))
eval_df["llm_term"] = eval_df["llm_sla"].apply(lambda x: x.get("lease_term_months"))
eval_df["llm_payment"] = eval_df["llm_sla"].apply(lambda x: x.get("monthly_payment"))

eval_df[["llm_apr", "llm_term", "llm_payment"]]

,llm_apr,llm_term,llm_payment
0,10.49%,24,959


In [16]:
# Ground truth values for sample contract (manual / known values)
eval_df["expected_apr"] = 10.49
eval_df["expected_term"] = 24
eval_df["expected_payment"] = 959

eval_df[[
    "llm_apr", "expected_apr",
    "llm_term", "expected_term",
    "llm_payment", "expected_payment"
]]

,llm_apr,expected_apr,llm_term,expected_term,llm_payment,expected_payment
0,10.49%,10.49,24,24,959,959


In [17]:
accuracy = {
    "APR Accuracy (%)": (eval_df["llm_apr"] == eval_df["expected_apr"]).mean() * 100,
    "Term Accuracy (%)": (eval_df["llm_term"] == eval_df["expected_term"]).mean() * 100,
    "Payment Accuracy (%)": (eval_df["llm_payment"] == eval_df["expected_payment"]).mean() * 100
}

accuracy

{'APR Accuracy (%)': np.float64(0.0),
 'Term Accuracy (%)': np.float64(100.0),
 'Payment Accuracy (%)': np.float64(100.0)}

In [18]:
# ===============================
# Week 4 – VIN Lookup (NHTSA API)
# ===============================

import requests

def fetch_vehicle_details_from_vin(vin):
    """
    Fetch vehicle make, model, year using NHTSA VIN Decode API
    """
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValuesExtended/{vin}?format=json"
    response = requests.get(url, timeout=10)
    data = response.json()

    if not data.get("Results"):
        return None

    result = data["Results"][0]

    return {
        "vin": vin,
        "make": result.get("Make"),
        "model": result.get("Model"),
        "year": result.get("ModelYear")
    }

In [19]:
def fetch_vehicle_recalls(make, model, year):
    """
    Fetch recall information using NHTSA Recall API
    """
    if not make or not model or not year:
        return []

    url = (
        "https://api.nhtsa.gov/recalls/recallsByVehicle"
        f"?make={make}&model={model}&modelYear={year}"
    )

    response = requests.get(url, timeout=10)
    data = response.json()

    recalls = data.get("results", [])

    return [
        {
            "campaign_number": r.get("NHTSACampaignNumber"),
            "summary": r.get("Summary"),
            "consequence": r.get("Consequence")
        }
        for r in recalls
    ]

In [20]:
test_vin = "1HGCM82633A004352"

vehicle_info = fetch_vehicle_details_from_vin(test_vin)
vehicle_info

{'vin': '1HGCM82633A004352',
 'make': 'HONDA',
 'model': 'Accord',
 'year': '2003'}

In [21]:
recalls = fetch_vehicle_recalls(
    vehicle_info["make"],
    vehicle_info["model"],
    vehicle_info["year"]
)

recalls[:2]

[{'campaign_number': '19V182000',
  'summary': 'Honda (American Honda Motor Co.) is recalling specific 2003 Acura 3.2CL, 2013-2016 ILX, 2013-2014 ILX Hybrid, 2003-2006 MDX, 2007-2016 RDX, 2002-2003 3.2TL, 2004-2006, and 2009-2014 TL, 2010-2013 ZDX and 2001-2007 and 2009 Honda Accord, 2001-2005 Civic, 2003-2005 Civic Hybrid, 2001-2005 Civic GX NGV, 2002-2007 and 2010-2011 CR-V, 2003-2011 Element, 2007 Fit, 2002-2004 Odyssey, 2003-2008 Pilot, and 2006-2014 Ridgeline vehicles.  The affected vehicles received a replacement driver air bag inflator as part of a previous Takata inflator recall remedy or a replacement driver air bag module containing the same inflator type as a service part.  Due to a manufacturing error, in the event of a crash necessitating deployment of the driver frontal air bag, these inflators may explode.',
  'consequence': 'An explosion of an inflator within the driver frontal air bag module may result in sharp metal fragments striking the driver, front seat passenger 

In [22]:
# ===============================
# Week 4 – Combine SLA + Vehicle Data
# ===============================

def build_final_contract_response(contract_id, sla_data, vin):
    vehicle = fetch_vehicle_details_from_vin(vin)

    recalls = []
    if vehicle:
        recalls = fetch_vehicle_recalls(
            vehicle["make"],
            vehicle["model"],
            vehicle["year"]
        )

    return {
        "contract_id": contract_id,
        "sla": sla_data,
        "vehicle": vehicle,
        "recalls": recalls
    }

In [23]:
# Internal end-to-end test

sample_contract = sla_json_records[0]
sample_vin = "1HGCM82633A004352"

final_response = build_final_contract_response(
    contract_id=sample_contract["contract_id"],
    sla_data=sample_contract["sla"],
    vin=sample_vin
)

final_response

{'contract_id': 0,
 'sla': {'interest_rate_apr': 10.49,
  'lease_term_months': 24,
  'monthly_payment': 959,
  'down_payment': None,
  'residual_value': None,
  'mileage_allowance': None,
  'overage_charge': None,
  'early_termination': None,
  'purchase_option': None,
  'maintenance_responsibility': None,
  'warranty_insurance': None,
  'penalties': None,
  'missing_or_ambiguous_clauses': ['down_payment',
   'residual_value',
   'mileage_allowance',
   'overage_charge',
   'early_termination',
   'purchase_option',
   'maintenance_responsibility',
   'warranty_insurance',
   'penalties']},
 'vehicle': {'vin': '1HGCM82633A004352',
  'make': 'HONDA',
  'model': 'Accord',
  'year': '2003'},
 'recalls': [{'campaign_number': '19V182000',
   'summary': 'Honda (American Honda Motor Co.) is recalling specific 2003 Acura 3.2CL, 2013-2016 ILX, 2013-2014 ILX Hybrid, 2003-2006 MDX, 2007-2016 RDX, 2002-2003 3.2TL, 2004-2006, and 2009-2014 TL, 2010-2013 ZDX and 2001-2007 and 2009 Honda Accord, 2001